## Read the file

In [1]:
from pathlib import Path 

# file_path = Path("../../data/whatsapp_chats/DummyData.txt")
file_path = Path("../../data/whatsapp_chats/WhatsApp Chat with Amie.txt")

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

len(lines)

4145

## Clean the conversation

In [2]:
import re

encryption_message = "Messages and calls are end-to-end encrypted. Only people in this chat can read, listen to, or share them. Learn more."
media_pattern = "<Media omitted>"
email_pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}'
url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
edited_message = "<This message was edited>"
deleted_message = "You deleted this message"
null_message = "null"
created_group_message = "created group"
added_you_to_group_message = "added you"
tagging_pattern = r'@[\w]+'


filtered_lines = []
for line in lines:
    if (
            encryption_message not in line and
            deleted_message not in line and
            null_message != line.split(" ")[-1] and
            media_pattern not in line and
            created_group_message not in line and
            added_you_to_group_message not in line and
            not re.search(email_pattern, line) and
            not re.search(url_pattern, line)
    ):
        line = line.replace(edited_message, "").strip()
        line = re.sub(tagging_pattern, "", line).strip()
        filtered_lines.append(line)

pattern = r'(\d{2}/\d{2}/\d{2,4}, \d{1,2}:\d{2}(?:\s?[ap]m)?) - (.*?): (.*?)(?=\n\d{2}/\d{2}/\d{2,4}, \d{1,2}:\d{2}(?:\s?[ap]m)? -|$)'

content = '\n'.join(filtered_lines)
messages = re.findall(pattern, content, re.DOTALL)

lines_removed = len(lines) - len(filtered_lines)
print(f"Lines removed: {lines_removed}")

Lines removed: 315


In [4]:
for m in messages: print(m)

('01/04/25, 4:47 pm', 'sampath', '🐨')
('01/04/25, 4:49 pm', 'Amie', 'Hey')
('01/04/25, 4:50 pm', 'sampath', 'You like stickers?')
('01/04/25, 4:50 pm', 'Amie', 'Yea I do')
('01/04/25, 4:50 pm', 'sampath', 'Like these 😂')
('01/04/25, 4:51 pm', 'Amie', 'Awww')
('01/04/25, 4:51 pm', 'sampath', 'Wanna see me now? Coming for break?')
('01/04/25, 4:51 pm', 'Amie', 'When will you leave?')
('01/04/25, 4:52 pm', 'sampath', '8')
('01/04/25, 4:52 pm', 'Amie', 'I was in bala just noe')
('01/04/25, 4:52 pm', 'sampath', 'Back to work now?')
('01/04/25, 4:52 pm', 'Amie', 'I have no work today')
('01/04/25, 4:52 pm', 'Amie', 'Yea come to my team')
('01/04/25, 4:52 pm', 'Amie', "It's chill")
('01/04/25, 4:53 pm', 'sampath', 'My team too')
('01/04/25, 4:53 pm', 'sampath', 'Okay tell when to meet')
('01/04/25, 4:53 pm', 'Amie', 'You are free now?')
('01/04/25, 4:53 pm', 'sampath', 'Yes')
('01/04/25, 4:53 pm', 'Amie', "Okay then I'll come")
('01/04/25, 4:53 pm', 'sampath', 'Okay i will be sitting opposite

## Create the dataset

### 1. Group messages by sender

If a conversation is structured as follows:  

```
User 1: Hey!  
User 1: How are you?  
User 2: I am fine  
User 2: And you?  
User 1: Good.  
```

We want to transform it into:  

```
User 1: Hey!\nHow are you? 
User 2: I am fine\nAnd you?  
User 1: Good  
```

In [5]:
grouped_messages = []

for _, sender, message in messages:
    if grouped_messages and grouped_messages[-1]["sender"] == sender:
        grouped_messages[-1]["message"] += "\n" + message
    else:
        grouped_messages.append({
            "sender": sender,
            "message": message
        })

len(grouped_messages)

2500

In [6]:
grouped_messages

[{'sender': 'sampath', 'message': '🐨'},
 {'sender': 'Amie', 'message': 'Hey'},
 {'sender': 'sampath', 'message': 'You like stickers?'},
 {'sender': 'Amie', 'message': 'Yea I do'},
 {'sender': 'sampath', 'message': 'Like these 😂'},
 {'sender': 'Amie', 'message': 'Awww'},
 {'sender': 'sampath', 'message': 'Wanna see me now? Coming for break?'},
 {'sender': 'Amie', 'message': 'When will you leave?'},
 {'sender': 'sampath', 'message': '8'},
 {'sender': 'Amie', 'message': 'I was in bala just noe'},
 {'sender': 'sampath', 'message': 'Back to work now?'},
 {'sender': 'Amie',
  'message': "I have no work today\nYea come to my team\nIt's chill"},
 {'sender': 'sampath', 'message': 'My team too\nOkay tell when to meet'},
 {'sender': 'Amie', 'message': 'You are free now?'},
 {'sender': 'sampath', 'message': 'Yes'},
 {'sender': 'Amie', 'message': "Okay then I'll come"},
 {'sender': 'sampath', 'message': 'Okay i will be sitting opposite of bala'},
 {'sender': 'Amie', 'message': 'Okay'},
 {'sender': 

### 2. Include special tokens

Each message follows this format:  
```
<|startoftext|>Sender<|separator|>Message<|endoftext|>
```

In [7]:
# Define special tokens
start_of_text_token = "<|startoftext|>"
end_of_text_token = "<|endoftext|>"
separator_token = "<|separator|>"

fine_tuning_data = []

for message in grouped_messages:
    sender = message["sender"]
    message_text = message["message"]
    input_sequence = f"{start_of_text_token}{sender}{separator_token}{message_text}{end_of_text_token}"
    fine_tuning_data.append(input_sequence)

len(fine_tuning_data)

2500

### 3. Save the data

In [9]:
import json

save_path = "../output/fine_tuning.json"
with open(save_path, 'w', encoding='utf-8') as f:
    json.dump(fine_tuning_data, f, ensure_ascii=False, indent=4)